# Naive Bayes Sentiment Analysis

## Install and Import Libraries

In [253]:
import pandas as pd
import nltk as nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt



## Load File, check data

In [254]:
filename = "mda_processed_sample_zeroshot_labeled.xlsx"
data = pd.read_excel(filename, sheet_name="before_stopword_zeroshot") # Use the sheet with stopwords NOT REMOVED


In [255]:
data.head()

,doc,sent_1,sent_2,sent_3,sent_4,sent_5,sent_6,sent_7,sent_8,sent_9,...,label_sent_442,label_sent_443,label_sent_444,label_sent_445,label_sent_446,label_sent_447,label_sent_448,label_sent_449,label_sent_450,label_sent_451
0,NVIDIA_10-K_2010-03-18_MDA.txt,"overview, our, company, nvidia, corporation, h...","expertise, in, programmable, gpus, has, led, t...","we, serve, the, entertainment, and, consumer, ...","during, the, last, several, fiscal, years, we,...","however, effective, with, the, first, quarter,...","our, gpu, business, is, comprised, primarily, ...","our, psb, is, comprised, of, our, nvidia, quad...","our, mcp, business, as, we, have, reported, it...","our, ion, family, of, products, addresses, the...",...,neutral,negative,negative,negative,neutral,neutral,neutral,neutral,neutral,neutral
1,NVIDIA_10-K_2011-03-16_MDA.txt,"overview, our, company, nvidia, corporation, i...","since, then, we, have, strived, to, set, new, ...","our, expertise, in, programmable, gpus, and, c...","we, are, strategically, investing, in, three, ...","we, serve, the, visual, computing, market, wit...","we, have, three, primary, financial, reporting...","during, fiscal, years, and, we, operated, and,...","however, during, the, first, quarter, of, fisc...","comparative, periods, presented, reflect, this...",...,neutral,neutral,negative,neutral,neutral,neutral,neutral,neutral,neutral,NaN
2,NVIDIA_10-K_2012-03-13_MDA.txt,"overview, our, company, nvidia, is, known, to,...","with, the, invention, of, the, gpu, we, brough...","today, we, reach, well, beyond, pc, graphics","our, energy-efficient, processors, power, a, b...","our, mobile, processors, are, used, in, cell, ...","pc, gamers, rely, on, our, gpus, to, enjoy, vi...","designers, use, them, to, create, visual, effe...","and, researchers, utilize, gpus, to, push, the...","nvidia, has, nearly, patents, granted, and, pe...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NVIDIA_10-K_2013-03-12_MDA.txt,"overview, our, company, nvidia, is, a, visual,...","in, a, world, increasingly, filled, with, visu...","visualization, transcends, cultural, and, lang...","we, have, long, been, known, to, millions, aro...","with, our, invention, of, the, gpu, we, introd...","today, we, reach, well, beyond, pc, graphics, ...","our, energy-efficient, processors, are, at, th...","pc, gamers, choose, our, gpus, by, name, to, e...","our, tegra, processors, power, smartphones, ta...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NVIDIA_10-K_2014-03-13_MDA.txt,"overview, our, company, nvidia, is, a, visual,...","in, a, world, increasingly, filled, with, visu...","our, strategy, is, to, be, the, world, leader,...","we, target, applications, in, each, of, the, m...","our, target, markets, are, gaming, design, and...","we, deploy, business, models, we, believe, are...","we, have, long, been, known, for, bringing, vi...","with, our, invention, of, the, gpu, we, introd...","today, we, reach, well, beyond, pc, graphics, ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Preprocessing

In [256]:
# Make a copy of the df first in case

df = data.copy()

In [257]:
sent_cols = [c for c in df.columns if c.startswith("sent_")]
label_cols = [c for c in df.columns if c.startswith("label_sent_")]

id_cols = [c for c in df.columns if c not in sent_cols + label_cols]

print("ID columns:", id_cols)

ID columns: ['doc']


In [258]:
# Create a new df so that for each row, it'll show the document name, 1 sentence and its label

rows = []

for _, row in df.iterrows():
    doc_name = row["doc"]
    
    for sent_col in sent_cols:
        sentence = row[sent_col]
        
        if pd.notna(sentence) and str(sentence).strip() != "":
            label_col = f"label_{sent_col}"   # sent_1 -> label_sent_1
            label = row[label_col]
            
            rows.append({
                "doc": doc_name,
                "sentence_id": sent_col,
                "sentence": sentence,
                "label": label
            })

long_df = pd.DataFrame(rows)
long_df.tail()

,doc,sentence_id,sentence,label
11940,NVIDIA_10-Q_2025-11-19_MDA.txt,sent_136,"other, than, the, contractual, obligations, de...",neutral
11941,NVIDIA_10-Q_2025-11-19_MDA.txt,sent_137,"refer, to, item, managements, discussion, and,...",neutral
11942,NVIDIA_10-Q_2025-11-19_MDA.txt,sent_138,"for, a, description, of, our, long-term, debt,...",neutral
11943,NVIDIA_10-Q_2025-11-19_MDA.txt,sent_139,"climate, change, to, date, there, has, been, n...",neutral
11944,NVIDIA_10-Q_2025-11-19_MDA.txt,sent_140,"adoption, of, new, and, recently, issued, acco...",neutral


In [259]:
long_df["sentence_fixed"] = (
    long_df["sentence"]
    .astype(str)
    .str.replace(",", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

long_df.head()

,doc,sentence_id,sentence,label,sentence_fixed
0,NVIDIA_10-K_2010-03-18_MDA.txt,sent_1,"overview, our, company, nvidia, corporation, h...",neutral,overview our company nvidia corporation helped...
1,NVIDIA_10-K_2010-03-18_MDA.txt,sent_2,"expertise, in, programmable, gpus, has, led, t...",positive,expertise in programmable gpus has led to brea...
2,NVIDIA_10-K_2010-03-18_MDA.txt,sent_3,"we, serve, the, entertainment, and, consumer, ...",neutral,we serve the entertainment and consumer market...
3,NVIDIA_10-K_2010-03-18_MDA.txt,sent_4,"during, the, last, several, fiscal, years, we,...",neutral,during the last several fiscal years we have o...
4,NVIDIA_10-K_2010-03-18_MDA.txt,sent_5,"however, effective, with, the, first, quarter,...",neutral,however effective with the first quarter of fi...


In [260]:
print("NaN values:")
print(long_df.isna().sum())

print("\nEmpty sentence values:")
print((long_df["sentence"].astype(str).str.strip() == "").sum())

NaN values:
doc               0
sentence_id       0
sentence          0
label             0
sentence_fixed    0
dtype: int64

Empty sentence values:
0


In [261]:
long_df["label"].value_counts()

label
neutral     9159
negative    1491
positive    1295
Name: count, dtype: int64

## Naive Bayes Stuff

In [262]:
# Split data into train and test

X_train, X_test, y_train, y_test = train_test_split(long_df['sentence_fixed'], long_df['label'], test_size=0.2, random_state=42, stratify=long_df['label'])

# X_train, X_test, y_train, y_test = train_test_split(long_df['sentence_fixed'], long_df['label'], test_size=0.2, random_state=42)


### 1. Bag of Words (Baseline)

In [263]:
vectorizer = CountVectorizer() # Init BoW Matrix


X_train_vector = vectorizer.fit_transform(X_train) # Fit and transform the training data
X_test_vector = vectorizer.transform(X_test) # Transform the testing data

In [264]:
nb_bow = MultinomialNB() # Init

nb_bow.fit(X_train_vector, y_train)


,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [265]:
y_pred1 = nb_bow.predict(X_test_vector)

# Print the classification report
print(classification_report(y_test, y_pred1))

              precision    recall  f1-score   support

    negative       0.62      0.75      0.68       298
     neutral       0.96      0.83      0.89      1832
    positive       0.50      0.89      0.64       259

    accuracy                           0.82      2389
   macro avg       0.69      0.82      0.74      2389
weighted avg       0.87      0.82      0.84      2389



#### 1a. Fine-tuning BoW

In [266]:
vectorizer_finetune = CountVectorizer(ngram_range=(1,2))

X_train_vector = vectorizer_finetune.fit_transform(X_train) # Fit and transform the training data
X_test_vector = vectorizer_finetune.transform(X_test)

In [267]:
nb_bow_fine = MultinomialNB() # Init

nb_bow_fine.fit(X_train_vector, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [268]:
y_pred2 = nb_bow_fine.predict(X_test_vector)

print(classification_report(y_test, y_pred2))

              precision    recall  f1-score   support

    negative       0.70      0.83      0.76       298
     neutral       0.97      0.86      0.91      1832
    positive       0.55      0.91      0.69       259

    accuracy                           0.86      2389
   macro avg       0.74      0.86      0.79      2389
weighted avg       0.89      0.86      0.87      2389



### 2. TF-IDF Vectorizer instead

In [269]:
tfidfvectorizer = TfidfVectorizer()

X_train_tfidf = tfidfvectorizer.fit_transform(X_train)
X_test_tfidf = tfidfvectorizer.transform(X_test)

In [270]:
tfidf_nb_model = MultinomialNB()


tfidf_nb_model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [271]:
y_pred_tfidf = tfidf_nb_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_tfidf))

              precision    recall  f1-score   support

    negative       0.93      0.39      0.55       298
     neutral       0.85      0.98      0.91      1832
    positive       0.78      0.41      0.54       259

    accuracy                           0.85      2389
   macro avg       0.85      0.60      0.67      2389
weighted avg       0.85      0.85      0.83      2389



## Pick a model, aggregate back

In [273]:
# Decide to go with fine-tuned BoW model

# Now run the model to predict all sentences

all_sentences_vec = vectorizer_finetune.transform(long_df["sentence_fixed"])
long_df["predicted_sentiment"] = nb_bow_fine.predict(all_sentences_vec)

In [277]:
long_df.head(20)

,doc,sentence_id,sentence,label,sentence_fixed,predicted_sentiment
0,NVIDIA_10-K_2010-03-18_MDA.txt,sent_1,"overview, our, company, nvidia, corporation, h...",neutral,overview our company nvidia corporation helped...,neutral
1,NVIDIA_10-K_2010-03-18_MDA.txt,sent_2,"expertise, in, programmable, gpus, has, led, t...",positive,expertise in programmable gpus has led to brea...,positive
2,NVIDIA_10-K_2010-03-18_MDA.txt,sent_3,"we, serve, the, entertainment, and, consumer, ...",neutral,we serve the entertainment and consumer market...,neutral
3,NVIDIA_10-K_2010-03-18_MDA.txt,sent_4,"during, the, last, several, fiscal, years, we,...",neutral,during the last several fiscal years we have o...,neutral
4,NVIDIA_10-K_2010-03-18_MDA.txt,sent_5,"however, effective, with, the, first, quarter,...",neutral,however effective with the first quarter of fi...,neutral
5,NVIDIA_10-K_2010-03-18_MDA.txt,sent_6,"our, gpu, business, is, comprised, primarily, ...",neutral,our gpu business is comprised primarily of our...,neutral
6,NVIDIA_10-K_2010-03-18_MDA.txt,sent_7,"our, psb, is, comprised, of, our, nvidia, quad...",neutral,our psb is comprised of our nvidia quadro prof...,neutral
7,NVIDIA_10-K_2010-03-18_MDA.txt,sent_8,"our, mcp, business, as, we, have, reported, it...",neutral,our mcp business as we have reported it throug...,neutral
8,NVIDIA_10-K_2010-03-18_MDA.txt,sent_9,"our, ion, family, of, products, addresses, the...",neutral,our ion family of products addresses the integ...,neutral
9,NVIDIA_10-K_2010-03-18_MDA.txt,sent_10,"our, cpb, is, comprised, of, our, tegra, mobil...",neutral,our cpb is comprised of our tegra mobile produ...,neutral
